# LLM Distillation

> A good teacher does more than give the correct answer—they explain their reasoning and correct mistakes as they happen. The relationship between large and small models is similar: rather than letting the small model figure everything out on its own, let the large model "teach" it.
>
> This section covers three distillation methods: Logit-based (learning output distributions), Feature-based (learning intermediate representations), and On-Policy (the large model grades in real time), and walks through the complete pipeline from distilling a large model down to a small one.

In the LLM context, the Teacher is the large model (e.g., GPT-4) and the Student is the small model (e.g., LLaMA 7B). The traditional approach is to have the Teacher write standard answers and have the Student learn from them.

However, after the Student has memorized the Teacher's outputs, quality drops once the Student generates on its own. The reason is that the Student's training data distribution differs from its own generation distribution—this gap is called distribution shift.

The earliest and most classic method is Logit-based distillation—having the Student learn not just the answer, but the Teacher's probability judgment for every token.

In [ ]:
import numpy as np

np.random.seed(42)

## 1. The Essence of Distillation

```
Plain SFT (Supervised Fine-Tuning):
  Teacher output: "Paris"
  Student learns: input "What is the capital of France?" -> output "Paris"
  Problem: only learned the answer, not the reasoning process

Distillation:
  Teacher output: probability for every token [Paris:0.9, London:0.05, Berlin:0.03, ...]
  Student learns: not only output "Paris", but also make the entire probability
                   distribution close to the Teacher's
  Benefit: Student learns the Teacher's "judgment"—knowing "Paris" is most likely,
           "London" is also plausible but lower probability
```

**Why is the probability distribution more valuable than the answer?**

When the Teacher says "Paris 90%, London 5%, Berlin 3%", it provides two extra pieces of information compared to just saying "Paris":
1. London and Berlin are also reasonable (just less correct)—this is called "dark knowledge"
2. The other hundreds of cities have near-zero probability—explicitly telling the Student which ones are wrong

This is the core insight behind Knowledge Distillation, proposed by Hinton in 2015.

In [ ]:
import numpy as np

# Interactive demo: hard labels vs. soft labels, comparing with real probability distributions
print("=== Hard Labels vs. Soft Labels ===")
print()

cities = ["Paris", "London", "Berlin", "Rome", "Madrid", "Tokyo", "Beijing", "Sydney"]
teacher_logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, -3.0, -4.0, -5.0])

# Hard labels (SFT): one-hot
hard_labels = np.zeros(len(cities))
hard_labels[0] = 1.0

print("Question: What is the capital of France?")
print()
print("Hard labels (SFT):")
for city, prob in zip(cities[:5], hard_labels[:5]):
    bar = "\u2588" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  -> Student only knows 'Paris is correct'")
print()

# Soft labels (distillation): probability distribution
temperature = 3.0
scaled_logits = teacher_logits / temperature
soft_labels = np.exp(scaled_logits) / np.exp(scaled_logits).sum()

print("Soft labels (distillation, T=3):")
for city, prob in zip(cities, soft_labels):
    bar = "\u2588" * int(prob * 40)
    print(f"  {city}: {prob:.1%} {bar}")
print("  -> Student learns:")
print("     1. Paris is the most correct")
print("     2. London, Berlin are also European capitals (similarity knowledge)")
print("     3. Tokyo, Beijing probability ~ 0 (completely irrelevant)")
print()

# Quantify information difference
hard_entropy = -np.sum(hard_labels * np.log(hard_labels + 1e-10))
soft_entropy = -np.sum(soft_labels * np.log(soft_labels + 1e-10))
print(f"Hard label entropy: {hard_entropy:.2f} bits")
print(f"Soft label entropy: {soft_entropy:.2f} bits")
print(f"-> Soft labels contain ~{soft_entropy:.1f} bits of information, far more than hard labels' {hard_entropy:.2f} bits!")

## 2. Method 1: Logit Distillation (The Classic Approach)

Make the Student's output probability distribution approximate the Teacher's output probability distribution.

**Loss formula**:

$$L = (1-\alpha) \cdot L_{CE}(S, y) + \alpha \cdot T^2 \cdot L_{KL}(S_T, T_T)$$

Where:
- $L_{CE}$: Cross-entropy between Student and the correct answer (ensures basic correctness)
- $L_{KL}$: KL divergence between Student and Teacher probability distributions (learns dark knowledge)
- $T$: Temperature parameter—the larger it is, the "softer" the Teacher's distribution (dark knowledge becomes more visible)
- $\alpha$: Weight balancing the two losses

**Effect of temperature T**:
```
T=1:  [0.90, 0.05, 0.03, 0.02]  <- Very sharp, dark knowledge not visible
T=5:  [0.40, 0.25, 0.20, 0.15]  <- Softened, dark knowledge emerges
T=20: [0.28, 0.26, 0.24, 0.22]  <- Too soft, becomes nearly uniform
```

T too large -> all tokens have similar probability -> no information
T too small -> no different from hard labels -> no dark knowledge
Typically T=3~10 works well.

In [ ]:
import numpy as np

# Demo: effect of temperature on probability distribution
print("=== Effect of Temperature T on Soft Labels ===")
print()

logits = np.array([5.0, 2.0, 1.0, 0.5, 0.1, 0.01, 0.001, 0.0001])
labels = ["Paris", "London", "Berlin", "Rome", "Madrid", "Vienna", "Prague", "Warsaw"]

for T in [1, 3, 10, 20]:
    scaled = logits / T
    probs = np.exp(scaled) / np.exp(scaled).sum()
    
    print(f"T={T:2d}: ", end="")
    for i in range(5):
        bar = "\u2588" * int(probs[i] * 50)
        print(f"{labels[i]}:{probs[i]:.3f} {bar}  ", end="")
    print()

print()
print("T=1:  Almost all Paris -> dark knowledge is hidden")
print("T=3:  London, Berlin have some probability -> dark knowledge emerges")
print("T=10: Distribution more uniform -> dark knowledge is rich but signal weakens")
print("T=20: Nearly uniform -> too little information")

## 3. Method 2: Data Distillation (The Most Practical)

Logit distillation requires the Student and Teacher to share the **same vocabulary**, which is nearly impossible in the LLM setting (GPT-4 and Qwen have different vocabularies).

**Data distillation bypasses this problem**: have the Teacher generate training data, and train the Student on this data with SFT.

```
Step 1: Collect prompts (from your business scenario)
  ["Write a poem about spring", "Explain quantum mechanics", "Translate: Hello -> Chinese", ...]

Step 2: Teacher generates high-quality answers for each prompt
  GPT-4: "Spring has arrived, all things come alive..."
  GPT-4: "Quantum mechanics is the study of subatomic particles..."

Step 3: Train Student on (prompt, teacher_answer) pairs
  Student does standard SFT, learning to imitate the Teacher's output style and quality
```

**Advantage**: No requirement for matching vocabularies; any Teacher can teach any Student.
**Disadvantage**: Only learns "what the answer looks like", not "the dark knowledge in the probability distribution".

**Advanced data distillation techniques**:
- **Multi-turn dialogue distillation**: Teacher generates multi-turn conversations, Student learns conversational rhythm
- **CoT distillation**: Teacher generates answers with reasoning steps, Student learns to reason
- **Rejection sampling**: Teacher generates multiple answers, only the best ones are kept for Student training

In [ ]:
# Simulate the data distillation pipeline
print("=== Data Distillation Pipeline Simulation ===")
print()

prompts = [
    "Explain what machine learning is",
    "Write a five-character poem about autumn",
    "Difference between list and tuple in Python",
]

# Simulate Teacher (GPT-4) generation
teacher_responses = [
    "Machine learning is a branch of AI that enables computers to learn patterns from data without explicit programming.",
    "Autumn wind sweeps fallen leaves, frost kills hundred flowers. Sitting alone by cold window, wondering if your clothes are warm.",
    "list is mutable (can add, remove, modify), tuple is immutable (cannot change after creation). list uses [], tuple uses ().",
]

print("Generate training data:")
for i, (prompt, response) in enumerate(zip(prompts, teacher_responses)):
    print(f"\n--- Sample {i+1} ---")
    print(f"User: {prompt}")
    print(f"Assistant: {response}")

print()
print(f"Generated {len(prompts)} training samples in total")
print("Student does SFT on this data, learning to imitate the Teacher's style.")
print()
print("In real projects, typically 10k~100k such samples are needed.")

## 4. Method 3: Feature Distillation (Advanced)

Not only learn the output distribution, but also learn intermediate layer representations.

```
Teacher (GPT-4, 96 layers):
  Layer 1 -> Layer 2 -> ... -> Layer 48 -> ... -> Layer 96 -> Output
                                     ^
Student (7B, 32 layers):             |  Make Student's Layer 16 output
  Layer 1 -> Layer 2 -> ... -> Layer 16 -> ... -> Layer 32 -> Output
                                     approximate Teacher's Layer 48
```

**Why is this effective?** Intermediate layers contain information about "how to understand this sentence", which is richer than the final output.

**Why is it rarely used?**
- Requires access to the Teacher's internal representations (not possible with closed-source models)
- Teacher and Student have different dimensions, requiring projection matrices for alignment
- High computational cost and memory consumption

Currently, the mainstream approach for LLM distillation is still **data distillation**; feature distillation appears more often in vision models (e.g., distilling ViT to CNN).

## 5. Hands-on: Distilling a 7B Model

The previous sections covered the principles of three distillation methods. Now let's string them together into a complete distillation pipeline. The entire process has four steps:

1. **Prepare training data**: Collect a batch of high-quality prompts covering the target domain—math reasoning, code generation, or general conversation
2. **Teacher generation**: Use the GPT-4 API to generate answers for each prompt, saving (prompt, teacher_answer) pairs
3. **Student training**: Train the Student model with Logit-based distillation loss (KL divergence) to learn the Teacher's output distribution
4. **Evaluation and comparison**: Compare the Student's scores before and after distillation using evaluation benchmarks

Below, each step comes with executable code. Even without a real GPT-4 API key, you can use a local MiniGPT to simulate the Teacher and Student roles and run through the complete pipeline.

In [ ]:
print("=== Hands-on: GPT-4 -> 7B Distillation Pipeline ===")
print()

steps = [
    ("Step 1: Choose a base model", [
        "Recommended: Qwen2.5-7B / Llama-3-8B / Mistral-7B",
        "Requirement: the base model should already have decent capability (not too weak)",
        "Choose an Instruct version (already knows how to follow instructions)",
    ]),
    ("Step 2: Collect prompts", [
        "Source 1: Your business data (real user questions)",
        "Source 2: Open-source datasets (OpenHermes, ShareGPT, WildChat)",
        "Source 3: Self-built -- use another LLM to generate diverse prompts",
        "Quantity: at least 5,000, recommended 50k~100k",
    ]),
    ("Step 3: Teacher generates answers", [
        "Use GPT-4 API to generate answers for each prompt",
        "system prompt: 'You are a helpful assistant. Please answer in detail and accurately.'",
        "temperature=0.7 (preserves some diversity)",
        "Cost: 50k prompts x ~500 tokens/prompt = 25M tokens = ~$250 (GPT-4o)",
    ]),
    ("Step 4: Data cleaning", [
        "Remove answers that are too short (<20 tokens)",
        "Remove refusals containing 'As an AI'",
        "Remove malformed answers",
        "Deduplicate (keep only one if similarity > 0.9)",
    ]),
    ("Step 5: SFT training", [
        "Tools: LLaMA-Factory / Axolotl / Firefly",
        "Format: ChatML or ShareGPT format",
        "Hyperparams: lr=2e-5, epochs=3, batch_size=128",
        "Hardware: 4xA100 (80G), training time ~6-12 hours",
    ]),
    ("Step 6: Evaluation", [
        "Run standard benchmarks with lm-eval (see notebook 23)",
        "Human evaluation on 100 business data samples",
        "Compare the gap between Student and Teacher",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

## 6. Distillation vs. OPD Comparison

| Dimension | Data Distillation | OPD (On-Policy Distillation) |
|:---|:---|:---|
| **When Teacher participates** | Before training (generates data) | During training (scores in real time) |
| **Student training data** | Standard answers written by Teacher | Answers written by Student itself |
| **Engineering complexity** | Low (just SFT) | High (requires rollout + real-time Teacher) |
| **Exposure Bias** | Yes (training and inference prefixes are inconsistent) | No (trained on real prefixes) |
| **Vocabulary requirement** | None | Needs alignment (or cross-tokenizer distillation) |
| **Cost** | Low (Teacher runs only once) | High (Teacher runs every training iteration) |
| **Best for** | Quick prototyping, limited budget | Pursuing maximum performance, has engineering team |

**Recommendation**: Start with data distillation for a quick first version; if the results are good enough, ship it. If results are insufficient, then consider OPD.

In [ ]:
# Distillation effectiveness simulation
print("=== Distillation Results Comparison (Simulated) ===")
print()

print("Assume Teacher is GPT-4, scoring 86.4 on MMLU")
print()

models = [
    ("GPT-4 (Teacher)", 86.4, "Baseline"),
    ("7B base (no distillation)", 64.2, "Raw capability"),
    ("7B + data distillation", 72.8, "+8.6 points"),
    ("7B + data distillation + CoT", 75.1, "+10.9 points"),
    ("7B + OPD", 77.3, "+13.1 points (but 10x cost)"),
]

print(f"{'Model':<35s} {'MMLU':>8s} {'Gain':>8s}")
print("-" * 55)
for name, score, note in models:
    improvement = score - 64.2
    print(f"{name:<35s} {score:>6.1f}  {improvement:>+6.1f}  ({note})")

print()
print("Conclusion: Data distillation offers the best cost-effectiveness; OPD performs best but costs more.")
print("In practice, data distillation + CoT distillation is the most commonly used combination.")

## 7. Common Questions About Distillation

| Question | Answer |
|:---|:---|
| **Can the Student surpass the Teacher?** | Theoretically no (the ceiling is the Teacher), but in practice, if the Student has additional training data, it may surpass the Teacher on specific tasks |
| **What does distillation lose?** | Creativity, long-tail knowledge, complex reasoning—these are the hardest parts of the Teacher's "dark knowledge" to distill |
| **How much data is needed?** | At least 5,000 samples; 50k+ recommended. Data quality > quantity |
| **What if Teacher and Student vocabularies differ?** | Use data distillation (Method 2), which requires no vocabulary matching |
| **Is RLHF still needed after distillation?** | Depends on the scenario. If the Teacher has already been aligned, the distilled Student is usually aligned as well |
| **Is multi-Teacher distillation feasible?** | Yes. GPT-4 teaches reasoning + Claude teaches writing + Gemini teaches multimodal -> stronger overall capability |

## Summary

- [x] **Essence of distillation**: Learn the Teacher's probability distribution (dark knowledge), not just the standard answer
- [x] **Logit distillation**: Make the Student's output distribution approximate the Teacher's; requires matching vocabularies
- [x] **Data distillation**: Teacher generates training data, Student does SFT; the most practical approach
- [x] **Feature distillation**: Learn intermediate layer representations; effective but engineering-heavy
- [x] **Hands-on pipeline**: Choose base model -> collect prompts -> Teacher generates -> clean -> SFT -> evaluate
- [x] **Distillation vs. OPD**: Data distillation is cost-effective; OPD performs best but costs more
- [x] **CoT distillation**: Have the Teacher generate answers with reasoning steps, so the Student learns to reason

**One-sentence summary**: Distillation = let the large model be the teacher, the small model be the student.
The most practical method is data distillation—have GPT-4 write 50k answers, and have the small model learn from them.

## Exercises

To be added.